# State Estimation: EKF, UKF, and Particle Filter

Implement and compare three state estimation algorithms for fixed-wing UAV navigation.

**State vector (13D):** `[px, py, pz, u, v, w, qw, qx, qy, qz, omega_x, omega_y, omega_z]`

**Control vector (3D):** `[throttle, elevator, rudder]`

**Measurement vector (10D):** `[px, py, pz, u, v, w, omega_x, omega_y, omega_z, airspeed]`

## 1. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import sqrtm, cholesky
from scipy.stats import multivariate_normal
import pandas as pd
import time

# Load dynamics model
%run dynamics.ipynb

## 2. Configuration Parameters

In [ ]:
# Simulation settings
dt = 0.01  # Time step (s)
trajectory_file = 'data/trajectory_example.csv'

# Initial covariance (moderate uncertainty)
P0 = np.diag([
    100, 100, 100,           # Position variance (m²)
    5, 5, 5,                 # Velocity variance (m/s)²
    0.1, 0.1, 0.1, 0.1,     # Quaternion variance
    0.2, 0.2, 0.2            # Angular rate variance (rad/s)²
])

# Process noise covariance (can enable control noise if desired)
USE_CONTROL_NOISE = True
Q_control = np.diag([0.01, 0.01, 0.01])**2  # Control noise variance
Q_state = np.zeros((13, 13))                 # State process noise (disabled)

# Measurement noise covariance (10x10)
R = np.diag([
    4.0, 4.0, 9.0,           # GPS position (m²): 2m, 2m, 3m std
    1.0, 1.0, 1.0,           # Doppler velocity (m/s)²: 1 m/s std each
    0.0025, 0.0025, 0.0025,  # Gyro (rad/s)²: 0.05 rad/s std
    2.25                     # Pitot airspeed (m/s)²: 1.5 m/s std
])

# Filter-specific parameters
ukf_alpha = 1e-3             # UKF spread parameter
ukf_beta = 2.0               # UKF distribution parameter (Gaussian optimal)
ukf_kappa = 0.0              # UKF secondary scaling
pf_num_particles = 1000      # Particle filter particle count

## 3. Dynamics and Measurement Models

Define the dynamics propagation and measurement functions needed for the filters.

In [ ]:
def dynamics(x, u, dt):
    """
    Propagate state through dynamics using RK4 integration.
    
    Args:
        x: Current state (13D) [px, py, pz, u, v, w, qw, qx, qy, qz, omega_x, omega_y, omega_z]
        u: Control input (3D) [throttle, elevator, rudder]
        dt: Time step
        
    Returns:
        x_next: Next state (13D)
    """
    # YOUR CODE HERE
    # Use the f(x, u) function from dynamics.ipynb
    # Implement RK4 integration:
    # k1 = f(x, u)
    # k2 = f(x + dt/2 * k1, u)
    # k3 = f(x + dt/2 * k2, u)
    # k4 = f(x + dt * k3, u)
    # x_next = x + dt/6 * (k1 + 2*k2 + 2*k3 + k4)
    pass

In [ ]:
def measurement_model(x):
    """
    Compute expected sensor measurements from state.
    
    Args:
        x: State vector (13D)
        
    Returns:
        z: Measurement vector (10D) [px, py, pz, u, v, w, omega_x, omega_y, omega_z, airspeed]
    """
    # YOUR CODE HERE
    # Extract position: x[0:3]
    # Extract velocity: x[3:6]
    # Extract angular rates: x[10:13]
    # Compute airspeed: sqrt(u^2 + v^2 + w^2)
    # Return concatenated measurement vector
    pass

In [ ]:
def measurement_jacobian(x):
    """
    Compute Jacobian of measurement model with respect to state.
    
    Args:
        x: State vector (13D)
        
    Returns:
        H: Measurement Jacobian (10x13)
    """
    # YOUR CODE HERE
    # Most entries are 0 or 1 (direct measurements)
    # First 9 measurements are direct:
    # - Position: H[0:3, 0:3] = I_3x3
    # - Velocity: H[3:6, 3:6] = I_3x3
    # - Angular rates: H[6:9, 10:13] = I_3x3
    # 
    # Airspeed (10th measurement) requires chain rule:
    # h_airspeed = sqrt(u^2 + v^2 + w^2)
    # dh/du = u / sqrt(u^2 + v^2 + w^2)
    # dh/dv = v / sqrt(u^2 + v^2 + w^2)
    # dh/dw = w / sqrt(u^2 + v^2 + w^2)
    # H[9, 3:6] = [dh/du, dh/dv, dh/dw]
    pass

## 4. Load and Process Trajectory Data

Load true trajectory from CSV and generate noisy measurements.

In [ ]:
# Load CSV
data = pd.read_csv(trajectory_file)

# Extract data
time_data = data['time'].values
states_true = data.iloc[:, 1:14].values      # Columns 1-13: state
controls = data.iloc[:, 14:17].values        # Columns 14-16: control

N_steps = len(time_data)

print(f"Loaded trajectory with {N_steps} time steps")
print(f"Duration: {time_data[-1]:.2f} seconds")
print(f"Time step: {np.mean(np.diff(time_data)):.4f} seconds")

In [ ]:
# Generate noisy measurements
np.random.seed(42)  # For reproducibility

measurements = np.zeros((N_steps, 10))
for k in range(N_steps):
    z_clean = measurement_model(states_true[k])
    measurements[k] = z_clean + np.random.multivariate_normal(np.zeros(10), R)

print("Generated noisy measurements")

In [ ]:
# Plot true trajectory and noisy measurements
# YOUR CODE HERE
# Create subplots showing:
# - 3D position (true vs measurements)
# - Velocity components (true vs measurements)
# - Angular rates (true vs measurements)
# - Airspeed (true vs measurements)

## 5. Results Storage

Initialize dictionary structure to store results from all filters.

In [ ]:
results = {
    'ekf': {
        'estimates': np.zeros((N_steps, 13)),
        'covariances': [],
        'rmse': {},
        'compute_time': 0.0
    },
    'ukf': {
        'estimates': np.zeros((N_steps, 13)),
        'covariances': [],
        'rmse': {},
        'compute_time': 0.0
    },
    'pf': {
        'estimates': np.zeros((N_steps, 13)),
        'rmse': {},
        'compute_time': 0.0
    }
}

ground_truth = {
    'states': states_true,
    'measurements': measurements,
    'controls': controls,
    'time': time_data
}

## 6. Extended Kalman Filter (EKF)

Implement EKF using linearized dynamics for prediction and update steps.

### 6.1 Initialize

In [ ]:
# YOUR CODE HERE
# Initialize EKF state estimate and covariance
# x_ekf = states_true[0] + some_initial_error
# P_ekf = P0.copy()

### 6.2 Prediction Step

In [ ]:
def ekf_predict(x, P, u, dt, Q):
    """
    EKF prediction using linearized dynamics.
    
    Args:
        x: Current state estimate (13D)
        P: Current covariance (13x13)
        u: Control input (3D)
        dt: Time step
        Q: Process noise covariance (13x13)
        
    Returns:
        x_pred: Predicted state
        P_pred: Predicted covariance
    """
    # YOUR CODE HERE
    # 1. Propagate state: x_pred = dynamics(x, u, dt)
    # 2. Compute state Jacobian F using F(x, u) from dynamics.ipynb
    #    Note: F is continuous-time, discretize using: F_d = I + F * dt (first-order)
    #    or use matrix exponential for higher accuracy
    # 3. Propagate covariance: P_pred = F_d @ P @ F_d.T + Q
    pass

### 6.3 Update Step

In [ ]:
def ekf_update(x_pred, P_pred, z, R):
    """
    EKF measurement update.
    
    Args:
        x_pred: Predicted state (13D)
        P_pred: Predicted covariance (13x13)
        z: Measurement (10D)
        R: Measurement noise covariance (10x10)
        
    Returns:
        x_est: Updated state estimate
        P_est: Updated covariance
    """
    # YOUR CODE HERE
    # 1. Compute predicted measurement: z_pred = measurement_model(x_pred)
    # 2. Compute measurement Jacobian: H = measurement_jacobian(x_pred)
    # 3. Innovation: y = z - z_pred
    # 4. Innovation covariance: S = H @ P_pred @ H.T + R
    # 5. Kalman gain: K = P_pred @ H.T @ inv(S)
    # 6. Update state: x_est = x_pred + K @ y
    # 7. Update covariance: P_est = (I - K @ H) @ P_pred
    pass

### 6.4 Run EKF

In [ ]:
# YOUR CODE HERE
# Loop through all time steps:
# 1. Apply ekf_predict() with current control input
# 2. Apply ekf_update() with current measurement
# 3. Store estimate in results['ekf']['estimates'][k]
# 4. Optionally store covariance in results['ekf']['covariances']
# 5. Track total computation time

# start_time = time.time()
# for k in range(N_steps):
#     ...
# results['ekf']['compute_time'] = time.time() - start_time

## 7. Unscented Kalman Filter (UKF)

Implement UKF using unscented transform for nonlinear propagation.

### 7.1 Sigma Point Generation

In [ ]:
def generate_sigma_points(x, P, alpha, beta, kappa):
    """
    Generate sigma points for unscented transform.
    
    Args:
        x: State mean (13D)
        P: State covariance (13x13)
        alpha: Spread parameter (typically 1e-3)
        beta: Distribution parameter (2 for Gaussian)
        kappa: Secondary scaling (typically 0)
        
    Returns:
        sigma_points: Array of sigma points (27x13 for 13D state)
        weights_mean: Weights for mean computation (27,)
        weights_cov: Weights for covariance computation (27,)
    """
    # YOUR CODE HERE
    # 1. Compute lambda: lambda = alpha^2 * (n + kappa) - n
    # 2. Compute matrix square root: sqrt_P = cholesky((n + lambda) * P)
    # 3. Generate 2n+1 sigma points:
    #    sigma_0 = x
    #    sigma_i = x + sqrt_P[:, i-1] for i=1,...,n
    #    sigma_i = x - sqrt_P[:, i-n-1] for i=n+1,...,2n
    # 4. Compute weights:
    #    w_mean_0 = lambda / (n + lambda)
    #    w_cov_0 = lambda / (n + lambda) + (1 - alpha^2 + beta)
    #    w_i = 1 / (2 * (n + lambda)) for i=1,...,2n (both mean and cov)
    pass

### 7.2 Prediction

In [ ]:
def ukf_predict(x, P, u, dt, Q, alpha, beta, kappa):
    """
    UKF prediction using unscented transform.
    
    Args:
        x: Current state estimate (13D)
        P: Current covariance (13x13)
        u: Control input (3D)
        dt: Time step
        Q: Process noise covariance (13x13)
        alpha, beta, kappa: UKF parameters
        
    Returns:
        x_pred: Predicted state mean
        P_pred: Predicted covariance
    """
    # YOUR CODE HERE
    # 1. Generate sigma points from current estimate
    # 2. Propagate each sigma point through dynamics
    # 3. Compute predicted mean: x_pred = sum(w_i * sigma_i_pred)
    # 4. Compute predicted covariance: P_pred = sum(w_i * (sigma_i_pred - x_pred) @ (sigma_i_pred - x_pred).T) + Q
    pass

### 7.3 Update

In [ ]:
def ukf_update(x_pred, P_pred, z, R, alpha, beta, kappa):
    """
    UKF measurement update using unscented transform.
    
    Args:
        x_pred: Predicted state (13D)
        P_pred: Predicted covariance (13x13)
        z: Measurement (10D)
        R: Measurement noise covariance (10x10)
        alpha, beta, kappa: UKF parameters
        
    Returns:
        x_est: Updated state estimate
        P_est: Updated covariance
    """
    # YOUR CODE HERE
    # 1. Generate sigma points from predicted state
    # 2. Transform sigma points through measurement model
    # 3. Compute predicted measurement mean: z_pred = sum(w_i * z_i)
    # 4. Compute innovation covariance: S = sum(w_i * (z_i - z_pred) @ (z_i - z_pred).T) + R
    # 5. Compute cross-covariance: P_xz = sum(w_i * (sigma_i - x_pred) @ (z_i - z_pred).T)
    # 6. Kalman gain: K = P_xz @ inv(S)
    # 7. Update state: x_est = x_pred + K @ (z - z_pred)
    # 8. Update covariance: P_est = P_pred - K @ S @ K.T
    pass

### 7.4 Run UKF

In [ ]:
# YOUR CODE HERE
# Loop through all time steps:
# 1. Apply ukf_predict()
# 2. Apply ukf_update()
# 3. Store results in results['ukf']
# 4. Track computation time

## 8. Particle Filter (PF)

Implement particle filter using Monte Carlo sampling.

### 8.1 Initialize Particles

In [ ]:
def initialize_particles(x0, P0, N):
    """
    Generate initial particle cloud from Gaussian distribution.
    
    Args:
        x0: Initial state estimate (13D)
        P0: Initial covariance (13x13)
        N: Number of particles
        
    Returns:
        particles: (N x 13) array of particles
        weights: (N,) array of uniform weights
    """
    # YOUR CODE HERE
    # 1. Sample N particles from N(x0, P0)
    # 2. Initialize weights uniformly: weights = 1/N
    pass

### 8.2 Prediction (Propagate Particles)

In [ ]:
def pf_predict(particles, u, dt, Q):
    """
    Propagate particles through dynamics with process noise.
    
    Args:
        particles: Current particles (N x 13)
        u: Control input (3D)
        dt: Time step
        Q: Process noise covariance (13x13)
        
    Returns:
        particles_pred: Propagated particles (N x 13)
    """
    # YOUR CODE HERE
    # For each particle i:
    #   1. Propagate through dynamics: x_i = dynamics(particles[i], u, dt)
    #   2. Add process noise: x_i += sample from N(0, Q)
    pass

### 8.3 Update (Compute Weights)

In [ ]:
def pf_update_weights(particles, weights, z, R):
    """
    Update particle weights based on measurement likelihood.
    
    Args:
        particles: Particles (N x 13)
        weights: Current weights (N,)
        z: Measurement (10D)
        R: Measurement noise covariance (10x10)
        
    Returns:
        weights_updated: Updated normalized weights (N,)
    """
    # YOUR CODE HERE
    # For each particle i:
    #   1. Compute expected measurement: z_i = measurement_model(particles[i])
    #   2. Compute likelihood: p(z | x_i) = exp(-0.5 * (z - z_i).T @ inv(R) @ (z - z_i))
    #      Or use multivariate_normal.pdf(z, mean=z_i, cov=R)
    #   3. Update weight: weights[i] *= likelihood
    # 4. Normalize weights: weights /= sum(weights)
    pass

### 8.4 Resampling

In [ ]:
def pf_resample(particles, weights):
    """
    Resample particles to avoid degeneracy.
    
    Args:
        particles: Particles (N x 13)
        weights: Normalized weights (N,)
        
    Returns:
        particles_resampled: Resampled particles (N x 13)
        weights_uniform: Uniform weights (N,) = 1/N
    """
    # YOUR CODE HERE
    # Implement systematic resampling:
    # 1. Compute cumulative sum of weights
    # 2. Generate N uniformly spaced samples in [0, 1]
    # 3. For each sample, find corresponding particle index
    # 4. Resample particles and reset weights to uniform
    # 
    # Alternatively, use multinomial resampling:
    # indices = np.random.choice(N, size=N, p=weights)
    # particles_resampled = particles[indices]
    pass

### 8.5 Run PF

In [ ]:
# YOUR CODE HERE
# Initialize particles and weights
# Loop through all time steps:
# 1. Apply pf_predict()
# 2. Apply pf_update_weights()
# 3. Compute state estimate: x_est = sum(weights[i] * particles[i])
# 4. Check effective sample size: N_eff = 1 / sum(weights^2)
# 5. If N_eff < threshold (e.g., N/2), apply pf_resample()
# 6. Store estimate in results['pf']
# 7. Track computation time

## 9. Particle Filter Variants

Space for advanced particle filter implementations.

### Auxiliary Particle Filter

In [ ]:
# Future implementation

### Regularized Particle Filter

In [ ]:
# Future implementation

### Rao-Blackwellized Particle Filter

In [ ]:
# Future implementation

## 10. Results Comparison

Visualize and analyze performance of all three filters.

### 10.1 Position Tracking

In [ ]:
# YOUR CODE HERE
# Create subplots for px, py, pz
# Plot ground truth, EKF, UKF, PF estimates
# Include legend and labels

### 10.2 Velocity Tracking

In [ ]:
# YOUR CODE HERE
# Create subplots for u, v, w (body frame velocities)
# Plot ground truth vs filter estimates

### 10.3 Attitude Tracking

In [ ]:
# YOUR CODE HERE
# Convert quaternions to Euler angles (roll, pitch, yaw)
# Plot Euler angles for all filters vs ground truth
# Hint: Use rotation library or implement quaternion to Euler conversion

### 10.4 Angular Rate Tracking

In [ ]:
# YOUR CODE HERE
# Create subplots for omega_x, omega_y, omega_z
# Plot ground truth vs filter estimates

### 10.5 RMSE Computation

In [ ]:
# YOUR CODE HERE
# For each filter and each state component:
# Compute RMSE = sqrt(mean((estimate - truth)^2))
# Store in results[filter_name]['rmse']
# Create comparison table or bar chart

### 10.6 Computational Cost

In [ ]:
# YOUR CODE HERE
# Create bar chart comparing computation times:
# - EKF: results['ekf']['compute_time']
# - UKF: results['ukf']['compute_time']
# - PF: results['pf']['compute_time']
# Also compute time per iteration for each filter

## 11. Analysis

Discussion and conclusions from the comparison study.

### Discussion Questions

1. **Accuracy**: Which filter achieved the lowest RMSE? Why might this be the case given the nonlinearity of the fixed-wing dynamics?

2. **Computational Cost**: How do the computation times compare? Is the increased accuracy of UKF/PF worth the computational overhead?

3. **Robustness**: How do the filters handle nonlinearity? Does the linearization in EKF cause significant errors?

4. **Practical Considerations**: 
   - When would you choose EKF over UKF/PF in a real UAV application?
   - How does measurement noise affect each filter?
   - What happens if process noise is increased?

5. **Extensions**: 
   - How might particle filter variants improve performance?
   - Could you combine strengths of different filters (e.g., EKF for linear parts, PF for highly nonlinear)?

In [ ]:
# YOUR ANALYSIS HERE
# Provide written responses and supporting visualizations